# Análise Exploratória de Dados — Rain in Australia

**PCO213 — Aprendizado de Máquina e Mineração de Dados | UNIFEI**

Este notebook constitui a **Etapa 2** do projeto. Toda a lógica de cálculo e geração de
figuras está encapsulada nos módulos `src/eda.py`, `src/visualization.py` e
`src/data_loader.py`. O notebook serve como camada narrativa e visual.

**Objetivo:** compreender a estrutura do dataset, identificar padrões, distribuições,
valores ausentes e outliers que orientarão as decisões de pré-processamento na Etapa 3.

---

**Figuras geradas:**
1. Distribuição da variável-alvo `RainTomorrow`
2. Percentual de valores ausentes por coluna
3. Histogramas das principais variáveis numéricas
4. Boxplots das principais variáveis numéricas (agrupados por RainTomorrow)
5. Heatmap de correlação de Pearson
6. Relação entre `RainToday` e `RainTomorrow`

In [ ]:
import sys
from pathlib import Path

# Localiza a raiz do projeto (verifica se src/ existe nos candidatos)
def _find_project_root() -> Path:
    for candidate in [Path('.'), Path('..')]:
        resolved = candidate.resolve()
        if (resolved / 'src').exists() and (resolved / 'requirements.txt').exists():
            return resolved
    raise FileNotFoundError(
        "Raiz do projeto não encontrada. "
        "Execute o Jupyter a partir de machine-learning-project/ ou notebooks/"
    )

_root = _find_project_root()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

print(f"Raiz do projeto: {_root}")

# Exibe figuras inline no notebook
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

---
## Seção 1 — Carregamento e Inspeção Inicial

In [ ]:
from src.data_loader import download_dataset, load_raw, basic_inspect

# Tenta download automático (Kaggle API); exibe instruções manuais se indisponível.
# Se o CSV já existir em data/raw/, o download é pulado.
download_dataset()

In [ ]:
df_raw = load_raw()
basic_inspect(df_raw)

---
## Seção 2 — Dicionário de Dados

Tabela com nome da variável, tipo, percentual de ausentes e relevância meteorológica.
Os campos `decisao` e `justificativa` serão preenchidos empiricamente ao longo desta análise.

In [ ]:
from src.data_loader import build_data_dictionary

data_dict = build_data_dictionary(df_raw)
data_dict

---
## Seção 3 — Distribuição da Variável-Alvo (Fig 1)

A variável-alvo `RainTomorrow` é binária: `Yes` (chove amanhã) e `No` (não chove).
Espera-se desbalanceamento entre as classes, com `No` sendo majoritário (~78%).
Esse desbalanceamento justifica:
- uso de `class_weight='balanced'` nos classificadores;
- avaliação primária por F1-score e ROC-AUC, não apenas accuracy;
- atenção especial ao Recall (custo do falso negativo).

In [ ]:
from src.eda import plot_target_distribution

fig1 = plot_target_distribution(df_raw)
# Figura salva automaticamente em outputs/figures/01_target_distribution.png

---
## Seção 4 — Análise de Valores Ausentes (Fig 2)

Colunas com alto percentual de ausentes são candidatas a remoção. A decisão final
considera a relevância meteorológica de cada variável e é documentada no dicionário
de dados. O limiar de candidatura à remoção está definido em `config.MISSING_DROP_THRESHOLD = 40%`.

In [ ]:
from src.eda import plot_missing_values

fig2 = plot_missing_values(df_raw)
# Figura salva em outputs/figures/02_missing_values.png

---
## Seção 5 — Estatísticas Descritivas

Média, mediana, desvio padrão, mínimo, máximo e quartis das variáveis numéricas.
A discrepância entre média e mediana pode indicar assimetria nas distribuições.

In [ ]:
from src.eda import descriptive_stats

stats = descriptive_stats(df_raw)
stats

---
## Seção 6 — Análise de Outliers (Método IQR)

Outliers identificados pelo método IQR: valores abaixo de Q1 − 1,5·IQR
ou acima de Q3 + 1,5·IQR. Os outliers não são removidos automaticamente;
a decisão de tratamento é discutida na metodologia do artigo.

In [ ]:
from src.eda import outlier_analysis

outliers = outlier_analysis(df_raw)
print("Colunas com maior percentual de outliers:")
outliers.head(10)

---
## Seção 7 — Histogramas das Variáveis Numéricas (Fig 3)

Distribuição de cada variável numérica com curva de densidade estimada (KDE).
Permite identificar assimetria, bimodalidade e concentração de valores.

In [ ]:
from src.eda import plot_numeric_histograms

fig3 = plot_numeric_histograms(df_raw)
# Figura salva em outputs/figures/03_numeric_histograms.png

---
## Seção 8 — Boxplots por RainTomorrow (Fig 4)

Boxplots agrupados por `RainTomorrow` (Yes/No) para visualizar diferenças de distribuição
entre as classes. Variáveis com separação clara entre grupos (ex.: Humidity3pm, Pressure3pm)
tendem a ser fortes preditores.

In [ ]:
from src.eda import plot_numeric_boxplots

fig4 = plot_numeric_boxplots(df_raw)
# Figura salva em outputs/figures/04_numeric_boxplots.png

---
## Seção 9 — Mapa de Correlação de Pearson (Fig 5)

Correlações lineares entre variáveis numéricas, incluindo a codificação binária de
`RainTomorrow`. Correlações altas com o alvo (positivas ou negativas) indicam
variáveis com maior poder discriminativo.

In [ ]:
from src.eda import plot_correlation_heatmap

fig5 = plot_correlation_heatmap(df_raw)
# Figura salva em outputs/figures/05_correlation_heatmap.png

---
## Seção 10 — RainToday × RainTomorrow (Fig 6)

Análise da relação entre `RainToday` e `RainTomorrow`. Dias em que choveu
tendem a ter maior probabilidade de chuva no dia seguinte — o que confere
à variável `RainToday` alto poder preditivo.

In [ ]:
from src.eda import plot_rain_today_vs_tomorrow

fig6 = plot_rain_today_vs_tomorrow(df_raw)
# Figura salva em outputs/figures/06_raintoday_vs_tomorrow.png

---
## Seção 11 — Sumário da EDA e Próximos Passos

Execute a célula abaixo para listar as figuras e tabelas geradas.

In [ ]:
from src.config import FIGURES_DIR, TABLES_DIR

print("=" * 55)
print("SUMÁRIO DA EDA — ARQUIVOS GERADOS")
print("=" * 55)

print("\n📊 Figuras (outputs/figures/):")
for f in sorted(FIGURES_DIR.glob("*.png")):
    print(f"   {f.name}")

print("\n📋 Tabelas (outputs/tables/):")
for f in sorted(TABLES_DIR.glob("*.csv")):
    print(f"   {f.name}")

print("\n" + "=" * 55)
print("PRÓXIMOS PASSOS — ETAPA 3")
print("=" * 55)
print("""
Com base na EDA:
  1. Decidir quais colunas remover (alto % ausentes + baixa relevância).
  2. Implementar src/preprocessing.py:
     - clean_data(): limpeza estrutural → data/processed/
     - build_preprocessor(): ColumnTransformer (sem leakage)
  3. Criar notebooks/02_modeling.ipynb para treinamento dos modelos.
""")